# 03 — Mô hình 3: PhoBERT (Transformer-based)

Fine-tune `vinai/phobert-base-v2` dạng **cross-encoder**: premise và hypothesis đi vào
cùng một chuỗi `<s> premise </s></s> hypothesis </s>`, nên self-attention nhìn thấy
cả hai câu cùng lúc. Đây là khác biệt then chốt so với TextCNN/BiLSTM ở notebook 02
(hai câu được mã hóa độc lập rồi mới ghép) — và là lý do Transformer thường bỏ xa
hai mô hình kia trên NLI.

**Bắt buộc:** PhoBERT được huấn luyện trên văn bản đã tách từ, nên phải chạy
`underthesea.word_tokenize` trước tokenizer. Bỏ bước này thường mất vài điểm accuracy —
notebook có nhánh ablation để đo chính xác con số đó.

Yêu cầu: GPU (Settings → Accelerator → GPU T4 x2 hoặc P100). Chạy `01_eda.ipynb` trước.

## Setup Kaggle — kéo code từ GitHub

Chạy cell này **trước tiên** trên Kaggle. Bỏ qua được khi chạy ở máy local.
Sửa code ở máy → `git push` → chạy lại cell này để lấy bản mới.

In [1]:
import os

REPO_URL = "https://github.com/dofu18/ViANLI_DL_NLP.git"

if os.path.exists("/kaggle/input"):
    !rm -rf /kaggle/working/repo
    !git clone -q $REPO_URL /kaggle/working/repo
    !cp -r /kaggle/working/repo/src /kaggle/working/
    !cp -r /kaggle/working/repo/configs /kaggle/working/
    !cp -r /kaggle/working/repo/data /kaggle/working/     # split cố định từ 01_eda
    print("src/:", sorted(os.listdir("/kaggle/working/src")))
else:
    print("Chạy local — bỏ qua bước clone.")

src/: ['data.py', 'evaluate.py', 'models.py', 'train.py', 'trainer.py']


In [2]:
# BẮT BUỘC: Kaggle không cài sẵn underthesea. Thiếu nó thì word_segment() raise
# và nhánh PhoBERT dừng ngay, thay vì âm thầm train trên text chưa tách từ.
!pip install -q underthesea "transformers>=4.44" "datasets>=2.21" accelerate

import json, os, sys, time

import numpy as np
import pandas as pd
import torch

ON_KAGGLE = os.path.exists("/kaggle/input")
ROOT = "/kaggle/working" if ON_KAGGLE else os.path.abspath("..")
sys.path.insert(0, os.path.join(ROOT, "src"))

FIG_DIR = os.path.join(ROOT, "outputs", "figures")
LOG_DIR = os.path.join(ROOT, "outputs", "logs")
CKPT_DIR = os.path.join(ROOT, "outputs", "checkpoints")
for d in (FIG_DIR, LOG_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)

from data import LABELS, label_to_id, load_splits, normalize, set_seed, word_segment
from models import build_transformer, count_params
from evaluate import (error_examples, full_report, hf_compute_metrics,
                      plot_confusion_matrix, plot_learning_curve)

SEED = 42
set_seed(SEED)
assert torch.cuda.is_available(), "Bật GPU trong Settings của Kaggle Notebook"
print(torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 68.7 MB/s eta 0:00:00
Tesla T4


## 1. Nạp split cố định

Cùng nguồn split với notebook 02 — đây là điều kiện của "nguyên tắc so sánh công bằng".

In [3]:
from datasets import Dataset, DatasetDict

SPLIT_DIR = os.path.join(ROOT, "data", "splits")
COLS = {"premise": "premise", "hypothesis": "hypothesis", "label": "label"}

if os.path.exists(os.path.join(SPLIT_DIR, "train.csv")):
    ds = DatasetDict({
        name: Dataset.from_pandas(
            pd.read_csv(os.path.join(SPLIT_DIR, f"{name}.csv"), encoding="utf-8")
              .fillna({"premise": "", "hypothesis": ""}))
        for name in ("train", "validation", "test")
    })
    cols = COLS
    print("Nạp từ data/splits/ (split cố định)")
else:
    ds, cols = load_splits(seed=SEED)
    print("CẢNH BÁO: chưa có data/splits/ — chạy 01_eda.ipynb trước")

{k: len(v) for k, v in ds.items()}

Nạp từ data/splits/ (split cố định)


{'train': 8012, 'validation': 1000, 'test': 1000}

## 2. Tokenize

`word_segment` tốn thời gian nên cache lại kết quả một lần, dùng chung cho cả nhánh
chính lẫn nhánh ablation.

In [4]:
MODEL_ID = "vinai/phobert-base-v2"
REVISION = "86cd7fd4c148980922ac11a2cf5e257f2ba639e1"   # pin commit hash (yêu cầu của đề)
MAX_LENGTH = 128     # chốt theo p95 của 01_eda (premise 51 + hypothesis 28 từ ~ 120 subword)

t0 = time.time()
seg_cache = {}

def prep(text, segment=True):
    text = normalize(text)
    if not segment:
        return text
    if text not in seg_cache:
        seg_cache[text] = word_segment(text)
    return seg_cache[text]

# Fail fast: PhoBERT-base-v2 giả định đầu vào ĐÃ tách từ. Bản chạy trước rơi vào
# fallback im lặng nên ablation no_wordseg trùng khít baseline (Δ = 0.000, vô nghĩa).
_probe = prep("Tọa đàm được tổ chức tại Hà Nội")
assert "_" in _probe, f"Tách từ KHÔNG hoạt động ({_probe!r}). Cài underthesea rồi restart kernel."
print("Tách từ OK:", _probe)

print("Ví dụ tách từ:")
print(" ", prep(ds["train"][cols["premise"]][0]))
print(" ", prep(ds["train"][cols["hypothesis"]][0]))

[word_segment] backend = underthesea
Tách từ OK: Tọa_đàm được tổ_chức tại Hà_Nội
Ví dụ tách từ:
  Tọa_đàm do Tổng_cục Du_lịch phối_hợp với báo_điện_tử VnExpress tổ_chức ngày 3/4 tại FLC Sầm_Sơn , Thanh_Hóa .
  Đầu tháng 4 có một buổi gặp_mặt trao_đổi , nói_chuyện thân_mật của Tổng_cục Du_lịch .


In [5]:
tokenizer, _ = build_transformer(MODEL_ID, revision=REVISION)

def make_encoded(segment=True, hypothesis_only=False, max_length=MAX_LENGTH):
    def tok(batch):
        n = len(batch[cols["label"]])
        premises = [""] * n if hypothesis_only else \
            [prep(x, segment) for x in batch[cols["premise"]]]
        out = tokenizer(premises,
                        [prep(x, segment) for x in batch[cols["hypothesis"]]],
                        truncation=True, max_length=max_length)
        out["labels"] = [label_to_id(y) for y in batch[cols["label"]]]
        return out
    return ds.map(tok, batched=True, remove_columns=ds["train"].column_names)

encoded = make_encoded()
print(f"Tokenize xong sau {time.time() - t0:.1f}s")

# Kiểm tra tỉ lệ bị cắt cụt — nếu cao thì phải tăng MAX_LENGTH
lens = [len(x) for x in encoded["train"]["input_ids"]]
print(f"độ dài subword: p50={np.percentile(lens, 50):.0f} "
      f"p95={np.percentile(lens, 95):.0f} max={max(lens)}")
print(f"tỉ lệ chạm trần {MAX_LENGTH}: {np.mean(np.array(lens) >= MAX_LENGTH):.2%}")
print(tokenizer.decode(encoded["train"][0]["input_ids"]))

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

...

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


Tokenize xong sau 42.9s
độ dài subword: p50=45 p95=72 max=128
tỉ lệ chạm trần 128: 0.04%
<s> Tọa_đàm do Tổng_cục Du_lịch phối_hợp với báo_điện_tử VnExpress tổ_chức ngày 3/4 tại FLC Sầm_Sơn , Thanh_Hóa . </s> </s> Đầu tháng 4 có một buổi gặp_mặt trao_đổi , nói_chuyện thân_mật của Tổng_cục Du_lịch . </s>


## 3. Hàm chạy một thí nghiệm

Cùng giao thức với notebook 02: chọn checkpoint tốt nhất theo **macro-F1 trên dev**,
test set chỉ chạy một lần ở cuối.

In [6]:
from transformers import (DataCollatorWithPadding, EarlyStoppingCallback,
                          Trainer, TrainingArguments)

BASE = dict(learning_rate=2e-5, batch_size=32, grad_accum=1, epochs=8,
            warmup_ratio=0.06, weight_decay=0.01, patience=3)

RESULTS = {}

def run(run_name, enc, model_id=MODEL_ID, **overrides):
    cfg = {**BASE, **overrides}
    set_seed(SEED)
    _, model = build_transformer(model_id, revision=REVISION)

    args = TrainingArguments(
        output_dir=os.path.join(CKPT_DIR, run_name),
        learning_rate=cfg["learning_rate"],
        per_device_train_batch_size=cfg["batch_size"],
        per_device_eval_batch_size=cfg["batch_size"] * 2,
        gradient_accumulation_steps=cfg["grad_accum"],
        num_train_epochs=cfg["epochs"],
        warmup_ratio=cfg["warmup_ratio"],
        weight_decay=cfg["weight_decay"],
        fp16=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_strategy="epoch",
        logging_dir=os.path.join(LOG_DIR, run_name),
        seed=SEED,
        report_to=[],
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=enc["train"], eval_dataset=enc["validation"],
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=hf_compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg["patience"])],
    )

    print(f"\n=== {run_name} | {count_params(model):,} tham số ===")
    torch.cuda.reset_peak_memory_stats()
    started = time.time()
    trainer.train()
    elapsed = time.time() - started

    logs = trainer.state.log_history
    losses = {int(r["epoch"]): r["loss"] for r in logs if "loss" in r}
    history = [{"epoch": int(r["epoch"]),
                "train_loss": losses.get(int(r["epoch"]), float("nan")),
                "val_macro_f1": r["eval_macro_f1"],
                "val_accuracy": r["eval_accuracy"]}
               for r in logs if "eval_macro_f1" in r]

    # history dạng dữ liệu (không chỉ dạng hình) — nb 05 và báo cáo cần file này
    with open(os.path.join(LOG_DIR, f"{run_name}_history.json"), "w",
              encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)

    pred = trainer.predict(enc["test"])
    y_true, y_pred = pred.label_ids, pred.predictions.argmax(-1)
    report = full_report(y_true, y_pred,
                         out_json=os.path.join(LOG_DIR, f"{run_name}_test.json"))
    plot_learning_curve(history, run_name,
                        os.path.join(FIG_DIR, f"{run_name}_curve.png"))
    plot_confusion_matrix(y_true, y_pred, run_name,
                          os.path.join(FIG_DIR, f"{run_name}_cm.png"))

    # thời gian suy luận cho §Chi phí tính toán
    t1 = time.time(); trainer.predict(enc["test"])
    ms_per_sample = (time.time() - t1) / len(enc["test"]) * 1000

    summary = {"run_name": run_name, "model_id": model_id,
               "accuracy": report["accuracy"], "macro_f1": report["macro_f1"],
               "weighted_f1": report["weighted_f1"],
               "params": count_params(model),
               "epochs_chạy": len(history),
               "train_seconds": round(elapsed, 1),
               "inference_ms_per_sample": round(ms_per_sample, 2),
               "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1024 ** 3, 2),
               "word_segment": enc is encoded}
    with open(os.path.join(LOG_DIR, f"{run_name}_summary.json"), "w",
              encoding="utf-8") as f:
        json.dump({**summary, "config": cfg}, f, ensure_ascii=False, indent=2)
    RESULTS[run_name] = {**summary, "_preds": (y_true, y_pred),
                         "_history": history}
    if run_name == "phobert_base_v2":
        best_dir = os.path.join(CKPT_DIR, run_name, "best")
        trainer.save_model(best_dir)
        tokenizer.save_pretrained(best_dir)
        print("Đã lưu:", best_dir)
    # không giữ trainer trong RESULTS: 4 bản PhoBERT cùng lúc là thừa VRAM/RAM
    del trainer, model
    torch.cuda.empty_cache()
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return summary

## 4. Fine-tune PhoBERT

Khoảng 8 epoch, early stopping patience 3. Trên T4 với batch 32 và `max_length=256`,
mỗi epoch thường mất vài phút — canh quota GPU.

In [7]:
run("phobert_base_v2", encoded)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is de


=== phobert_base_v2 | 135,000,579 tham số ===


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,2.161270,2.101190,0.444000,0.355994,0.356193
2,2.028575,2.174553,0.451000,0.409389,0.409561
3,1.772176,2.241843,0.450000,0.445890,0.445971
4,1.510138,2.385502,0.464000,0.458806,0.458903
5,1.266863,2.552530,0.467000,0.461762,0.461855
6,1.098233,2.716874,0.466000,0.462263,0.462350
7,0.957773,2.880844,0.446000,0.443727,0.443813
8,0.864016,2.946600,0.453000,0.451753,0.451830


...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu: /kaggle/working/outputs/checkpoints/phobert_base_v2/best
{
  "run_name": "phobert_base_v2",
  "model_id": "vinai/phobert-base-v2",
  "accuracy": 0.47,
  "macro_f1": 0.4659422981501755,
  "weighted_f1": 0.4660288420398706,
  "params": 135000579,
  "epochs_chạy": 8,
  "train_seconds": 655.7,
  "inference_ms_per_sample": 3.02,
  "peak_vram_gb": 3.98,
  "word_segment": true
}


{'run_name': 'phobert_base_v2',
 'model_id': 'vinai/phobert-base-v2',
 'accuracy': 0.47,
 'macro_f1': 0.4659422981501755,
 'weighted_f1': 0.4660288420398706,
 'params': 135000579,
 'epochs_chạy': 8,
 'train_seconds': 655.7,
 'inference_ms_per_sample': 3.02,
 'peak_vram_gb': 3.98,
 'word_segment': True}

In [8]:
# (Việc lưu model tốt nhất đã chuyển vào trong run() để giải phóng VRAM ngay
#  sau mỗi lần chạy — xem cell định nghĩa run() ở trên.)
print("best checkpoint:", os.path.join(CKPT_DIR, "phobert_base_v2", "best"))

best checkpoint: /kaggle/working/outputs/checkpoints/phobert_base_v2/best


## 5. Ablation

| Nhánh | Câu hỏi |
|---|---|
| `no_wordseg` | Bỏ tách từ mất bao nhiêu điểm? (PhoBERT pretrain trên văn bản đã tách từ) |
| `hyponly` | PhoBERT đoán được nhãn khi không thấy premise? So với BiLSTM ở notebook 02 |
| `lr_5e-5` | Learning rate ảnh hưởng thế nào? |

Mỗi nhánh tốn thêm một lần train — bỏ bớt nếu hết quota, nhưng phải ghi rõ trong báo cáo.

In [9]:
run("phobert_no_wordseg", make_encoded(segment=False))

Map:   0%|          | 0/8012 [00:00<?, ? examples/s]

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{
  "run_name": "phobert_no_wordseg",
  "model_id": "vinai/phobert-base-v2",
  "accuracy": 0.421,
  "macro_f1": 0.42092556121155855,
  "weighted_f1": 0.4209864538321652,
  "params": 135000579,
  "epochs_chạy": 8,
  "train_seconds": 771.8,
  "inference_ms_per_sample": 3.49,
  "peak_vram_gb": 4.0,
  "word_segment": false
}


{'run_name': 'phobert_no_wordseg',
 'model_id': 'vinai/phobert-base-v2',
 'accuracy': 0.421,
 'macro_f1': 0.42092556121155855,
 'weighted_f1': 0.4209864538321652,
 'params': 135000579,
 'epochs_chạy': 8,
 'train_seconds': 771.8,
 'inference_ms_per_sample': 3.49,
 'peak_vram_gb': 4.0,
 'word_segment': False}

In [10]:
run("phobert_hyponly", make_encoded(hypothesis_only=True))

Map:   0%|          | 0/8012 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is de

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{
  "run_name": "phobert_hyponly",
  "model_id": "vinai/phobert-base-v2",
  "accuracy": 0.416,
  "macro_f1": 0.40925291704278316,
  "weighted_f1": 0.40933057512050475,
  "params": 135000579,
  "epochs_chạy": 6,
  "train_seconds": 281.1,
  "inference_ms_per_sample": 1.62,
  "peak_vram_gb": 2.78,
  "word_segment": false
}


{'run_name': 'phobert_hyponly',
 'model_id': 'vinai/phobert-base-v2',
 'accuracy': 0.416,
 'macro_f1': 0.40925291704278316,
 'weighted_f1': 0.40933057512050475,
 'params': 135000579,
 'epochs_chạy': 6,
 'train_seconds': 281.1,
 'inference_ms_per_sample': 1.62,
 'peak_vram_gb': 2.78,
 'word_segment': False}

In [11]:
run("phobert_lr5e-5", encoded, learning_rate=5e-5)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is de


=== phobert_lr5e-5 | 135,000,579 tham số ===


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,2.148693,2.071978,0.474000,0.442829,0.442946
2,1.882146,2.188366,0.453000,0.447836,0.447909
3,1.454338,2.309808,0.473000,0.472023,0.472087
4,1.032083,2.806524,0.465000,0.464454,0.464518
5,0.678314,3.366523,0.438000,0.438159,0.438204
6,0.470820,3.632010,0.456000,0.452373,0.452449


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{
  "run_name": "phobert_lr5e-5",
  "model_id": "vinai/phobert-base-v2",
  "accuracy": 0.45,
  "macro_f1": 0.4478718872411897,
  "weighted_f1": 0.4479367589821344,
  "params": 135000579,
  "epochs_chạy": 6,
  "train_seconds": 493.8,
  "inference_ms_per_sample": 3.01,
  "peak_vram_gb": 3.98,
  "word_segment": true
}


{'run_name': 'phobert_lr5e-5',
 'model_id': 'vinai/phobert-base-v2',
 'accuracy': 0.45,
 'macro_f1': 0.4478718872411897,
 'weighted_f1': 0.4479367589821344,
 'params': 135000579,
 'epochs_chạy': 6,
 'train_seconds': 493.8,
 'inference_ms_per_sample': 3.01,
 'peak_vram_gb': 3.98,
 'word_segment': True}

## 6. Tổng hợp

In [12]:
table = pd.DataFrame([
    {k: v for k, v in r.items() if not k.startswith("_")}
    for r in RESULTS.values()
]).sort_values("accuracy", ascending=False)
display(table.round(4))
table.to_csv(os.path.join(LOG_DIR, "phobert_results.csv"), index=False)

# So với kết quả CNN/RNN nếu notebook 02 đã chạy
prev = os.path.join(LOG_DIR, "cnn_rnn_results.csv")
if os.path.exists(prev):
    display(pd.concat([pd.read_csv(prev), table], ignore_index=True)
              .sort_values("accuracy", ascending=False)
              [["run_name", "accuracy", "macro_f1", "params", "train_seconds"]]
              .round(4))

,run_name,model_id,accuracy,macro_f1,weighted_f1,params,epochs_chạy,train_seconds,inference_ms_per_sample,peak_vram_gb,word_segment
0,phobert_base_v2,vinai/phobert-base-v2,0.470,0.4659,0.4660,135000579,8,655.7,3.02,3.98,True
3,phobert_lr5e-5,vinai/phobert-base-v2,0.450,0.4479,0.4479,135000579,6,493.8,3.01,3.98,True
1,phobert_no_wordseg,vinai/phobert-base-v2,0.421,0.4209,0.4210,135000579,8,771.8,3.49,4.00,False
2,phobert_hyponly,vinai/phobert-base-v2,0.416,0.4093,0.4093,135000579,6,281.1,1.62,2.78,False


In [13]:
# Lưu dự đoán trên test để 05_analysis.ipynb phân tích lỗi chéo giữa các mô hình
pred_dir = os.path.join(LOG_DIR, "predictions")
os.makedirs(pred_dir, exist_ok=True)
for name, r in RESULTS.items():
    np.save(os.path.join(pred_dir, f"{name}.npy"), r["_preds"][1])
y_true_path = os.path.join(pred_dir, "y_true.npy")
y_true_new = np.asarray(RESULTS[list(RESULTS)[0]]["_preds"][0])
if os.path.exists(y_true_path):
    assert np.array_equal(np.load(y_true_path), y_true_new), (
        "y_true lệch so với file đã lưu — thứ tự test set không nhất quán giữa các "
        "mô hình, mọi so sánh chéo ở nb 05 sẽ sai.")
else:
    np.save(y_true_path, y_true_new)
print(sorted(os.listdir(pred_dir)))

['phobert_base_v2.npy', 'phobert_hyponly.npy', 'phobert_lr5e-5.npy', 'phobert_no_wordseg.npy', 'y_true.npy']


In [14]:
best = table.iloc[0]["run_name"]
y_true, y_pred = RESULTS[best]["_preds"]
print(f"Tốt nhất: {best}")
print("Phân bố dự đoán:",
      {LABELS[i]: int((y_pred == i).sum()) for i in range(len(LABELS))})

pd.set_option("display.max_colwidth", 100)
pd.DataFrame(error_examples(ds["test"], cols, y_true, y_pred, n=10))

Tốt nhất: phobert_base_v2
Phân bố dự đoán: {'entailment': 390, 'neutral': 292, 'contradiction': 318}


,premise,hypothesis,gold,pred
0,"Sáng 23/5, ông Trần Văn Vịnh, Chủ tịch UBND phường Mai Động, cho biết lý do phong tỏa là nam sin...",Theo thông tin một công chức Nhà nước ở phường Mai Động đã có ít nhất 11 nam sinh đã bị mắc Covi...,contradiction,entailment
1,"Sau khi xem xét camera giám sát trong khu vực, cảnh sát nhận định nạn nhân đã bị cướp tài sản và...",Cảnh sát đã phán đoán bằng nhận định sắc bén sau khi xem camera an ninh.,neutral,entailment
2,"3h sáng cùng ngày, trái tim của của người hiến đã đập lại trong lồng ngực chàng trai.","3am, một người được tái sinh.",contradiction,entailment
3,"Ông Phạm Cao Vỹ, chủ tịch Hiệp hội Du lịch Sa Pa chia sẻ, nhận được lời kêu gọi từ hiệp hội, rất...",Ông Phạm Cao Vỹ được yêu thích.,neutral,entailment
4,"Theo Vogue, đôi dép của Questlove mạ vàng 24 carat, giúp nhạc sĩ Mỹ tạo nét khác biệt.",Đôi dép của nhạc sĩ tên Mỹ được làm từ vàng 24 carat.,contradiction,neutral
5,"Đang nóng ruột vì chưa biết làm thế nào để bay về sớm với hai con gái bị tai nạn, anh Trần Văn H...",Anh Trần Văn Hải không có ý định hỏi vé từ một người lạ.,neutral,entailment
6,"Nguyễn Xuân Thành, 17 tuổi, thừa nhận có xích mích với cụ bà 88 tuổi sống gần nhà nên sát hại, t...",Cụ bà 88 tuổi đã tử vong.,entailment,neutral
7,Hàng ngàn ly sữa đã được cộng đồng chung tay cùng Vinamilk và Quỹ sữa Vươn cao Việt Nam góp tặng...,Vinamilk đã kêu gọi cộng đồng cùng Quỹ sữa Vươn cao Việt Nam quyên góp cho chiến dịch thiện nguyện.,neutral,entailment
8,Cảnh sát thành phố Bhopal hôm 13/5 cho biết sự việc gây sốc xảy ra đầu tháng 4 và được công bố s...,Sự việc được công bố vào tháng 5.,entailment,contradiction
9,Các dòng chip xử lý đầu tiên của hai công ty dự kiến sẽ ra mắt vào cuối 2021 hoặc đầu 2022.,Vào cuối năm 2021 thị trường công nghệ sẽ đón nhận dòng chip xử lý đầu tiên của công ty Trung Quốc.,neutral,entailment


## Ghi chú cho Kaggle

- Dùng **Save Version → Save & Run All** để chạy nền; session tương tác 12h dễ ngắt giữa chừng.
- `save_total_limit=1` để `/kaggle/working` không vượt giới hạn 20GB.
- Nếu OOM: hạ `batch_size` xuống 16 và đặt `grad_accum=2` (effective batch giữ nguyên 32).
- Nếu notebook không có Internet: upload PhoBERT thành Kaggle Dataset rồi đặt
  `MODEL_ID = "/kaggle/input/phobert-base-v2"`.

## Cần điền vào báo cáo

- Bảng cấu hình mô hình 3: model ID, revision, số layer/head/hidden, số tham số, chiến lược fine-tune.
- Bảng siêu tham số (cột Transformer).
- Hình `phobert_base_v2_curve.png`, `phobert_base_v2_cm.png`.
- Bảng ablation từ `phobert_results.csv` — đặc biệt con số **mất bao nhiêu điểm khi bỏ tách từ**.
- Chênh lệch PhoBERT so với TextCNN/BiLSTM → luận điểm về vai trò của cross-attention.